Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import itertools

In [2]:
import requests
import re
import json
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
from pathlib import Path
from collections import deque

In [3]:
URL_MAP_FILE = "Crawled Contents/url_map.json"

In [4]:
URLS = [
    # Practical legal guidance
    ("https://www.washingtonlawhelp.org/en",
     "Crawled Contents/washingtonlawhelp"),

    # Washington statutes
    ("https://app.leg.wa.gov/rcw/",
     "Crawled Contents/washington/rcw"),

    # Court rules/forms
    ("https://www.courts.wa.gov/court_rules/",
     "Crawled Contents/wacourts/rules"),

    ("https://www.courts.wa.gov/forms/",
     "Crawled Contents/wacourts/forms"),

    # Local court systems
    ("https://www.seattle.gov/courts/",
     "Crawled Contents/seattle/courts"),

    ("https://kingcounty.gov/en/court/superior-court",
     "Crawled Contents/kingcounty/superior-court"),

    ("https://districtcourt.kingcounty.gov/",
     "Crawled Contents/kingcounty/districtcourt"),

    # Municipal code
    ("https://library.municode.com/wa/seattle/codes/municipal_code",
     "Crawled Contents/seattle/municipal-code"),

    # Immigration law
    ("https://www.uscis.gov/policy-manual",
     "Crawled Contents/uscis/policy-manual"),
]

In [5]:
def normalize_url(url):
    parsed = urlparse(url)
    normalized = parsed._replace(
        scheme=parsed.scheme.lower(),
        netloc=parsed.netloc.lower(),
        path=parsed.path.rstrip("/"),
        query="",
        fragment=""
    )
    return urlunparse(normalized)

def is_useful_page(url):
    skip_patterns = ["searchresults", "search?", "?page=", "?terms=", "?q="]
    return not any(p in url for p in skip_patterns)

In [ ]:
global_visited = set()

def crawl(seed_url, output_dir="Crawled Contents", max_depth=3, max_pages=500):
    domain   = urlparse(seed_url).netloc
    save_dir = Path(output_dir)
    save_dir.mkdir(exist_ok=True, parents=True)

    queue = deque([(seed_url, 0)])

    while queue and len(global_visited) < max_pages:
        url, depth = queue.popleft()
        norm_url   = normalize_url(url)

        if norm_url in global_visited or depth > max_depth:
            continue

        try:
            resp = requests.get(url, timeout=10)
            if "text/html" not in resp.headers.get("Content-Type", ""):
                continue

            global_visited.add(norm_url)

            filename = re.sub(r'[\\/:*?"<>|&=]', '_', norm_url) + ".html"
            filepath = str((save_dir / filename).resolve())

            # Skip if already crawled
            if Path(filepath).exists():
                global_visited.add(norm_url)
                url_map[filepath] = url
                print(f"  [SKIP - already crawled] {url}")
                continue
            
            (save_dir / filename).write_text(resp.text, encoding="utf-8", errors="ignore")
            url_map[filepath] = url

            if depth < max_depth:
                soup = BeautifulSoup(resp.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    next_url  = urljoin(url, a["href"]).split("#")[0]
                    norm_next = normalize_url(next_url)
                    if (urlparse(next_url).netloc == domain
                            and norm_next not in global_visited
                            and is_useful_page(next_url)):
                        queue.append((next_url, depth + 1))

            print(f"  [{len(global_visited)}] {url}")

        except Exception as e:
            print(f"  SKIP {url}: {e}")

In [8]:
for url, out_dir in URLS:
    print(f"\nCrawling {url} → {out_dir}")
    crawl(url, output_dir=out_dir)

Path("Crawled Contents").mkdir(exist_ok=True)
with open(URL_MAP_FILE, "w") as f:
    json.dump(url_map, f, indent=2)

print(f"\nDone. {len(global_visited)} pages crawled, {len(url_map)} URLs mapped.")


Crawling https://www.washingtonlawhelp.org/en → Crawled Contents/washingtonlawhelp
  [SKIP - already crawled] https://www.washingtonlawhelp.org/en

Crawling https://app.leg.wa.gov/rcw/ → Crawled Contents/washington/rcw
  [SKIP - already crawled] https://app.leg.wa.gov/rcw/

Crawling https://www.courts.wa.gov/court_rules/ → Crawled Contents/wacourts/rules
  [SKIP - already crawled] https://www.courts.wa.gov/court_rules/

Crawling https://www.courts.wa.gov/forms/ → Crawled Contents/wacourts/forms
  [SKIP - already crawled] https://www.courts.wa.gov/forms/

Crawling https://www.seattle.gov/courts/ → Crawled Contents/seattle/courts
  [SKIP - already crawled] https://www.seattle.gov/courts/

Crawling https://kingcounty.gov/en/court/superior-court → Crawled Contents/kingcounty/superior-court
  [SKIP - already crawled] https://kingcounty.gov/en/court/superior-court

Crawling https://districtcourt.kingcounty.gov/ → Crawled Contents/kingcounty/districtcourt
  [SKIP - already crawled] https://d

In [9]:
import jdk4py
import os
from pathlib import Path

# Set Java env vars
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)

java_home = Path(jdk4py.JAVA_HOME)
jvm_paths = list(java_home.rglob("jvm.dll"))
if jvm_paths:
    os.environ["JVM_PATH"] = str(jvm_paths[0])
    print(f"JVM found: {jvm_paths[0]}")

# Import and init PyTerrier
import pyterrier as pt
if not pt.java.started():
    pt.java.init()

JVM found: c:\Users\Jia Chen\Downloads\Projects\CISC-portal\server\ml_scripts\.venv\Lib\site-packages\jdk4py\java-runtime\bin\server\jvm.dll


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [10]:
dirs = [out_dir for _, out_dir in URLS]

all_files = [str(Path(f)) for f in itertools.chain.from_iterable(
    pt.io.find_files(d) for d in dirs
)]

all_files[:5]

['Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org.html',
 'Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_.html',
 'Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_accessibility-statement.html',
 'Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_cdn-cgi_l_email-protection.html',
 'Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_en.html']

In [11]:
index_path = os.path.join(os.getcwd(), "index")
os.makedirs(index_path, exist_ok=True)

indexer = pt.FilesIndexer(index_path, verbose=True)
indexref = indexer.index(all_files)

In [12]:
index = pt.IndexFactory.of(indexref)
print(index.getCollectionStatistics().toString())

Number of documents: 2014
Number of terms: 20981
Number of postings: 1006716
Number of fields: 0
Number of tokens: 4398188
Field names: []
Positions:   false



In [13]:
LEGAL_CATEGORIES = {
    "Consumer / Finance": {
        "Bankruptcy/Debtor Relief": 1,
        "Collections/Repo/Garnishment": 2,
        "Contracts/Warranties": 3,
        "Collection Practices/Creditor Harass.": 4,
        "Predatory Lending Practices": 5,
        "Loans/Installment Purchases": 6,
        "Public Utilities": 7,
        "Unfair Sales Practices": 8,
        "Other Consumer / Finance": 9,
    },
    "Education": {
        "Education": 11,
        "Discipline": 12,
        "Special Education/Learning Dis.": 13,
        "Access": 14,
        "Vocational Education": 15,
        "Student Financial Aid": 16,
        "Other Education": 19,
    },
    "Employment": {
        "Employment Discrimination": 21,
        "Wage Claims and Other FLSA": 22,
        "Earned Income Tax Credit": 23,
        "Taxes": 24,
        "Employee Rights": 25,
        "Agricultural Workers Issues": 26,
        "Other Employment": 29,
    },
    "Family": {
        "Adoption": 30,
        "Child Custody/Visitation": 31,
        "Divorce/Separation": 32,
        "Adult Guardianship": 33,
        "Name Change": 34,
        "Paternal Rights Termination": 35,
        "Paternity": 36,
        "Domestic Abuse": 37,
        "Child Support": 38,
        "Other Family Law": 39,
    },
    "Juvenile": {
        "Juvenile Delinquent": 41,
        "Dependency (Abuse/Neglect)": 42,
        "Emancipation": 43,
        "Guardianship/3rd party custody": 44,
        "Other Juvenile": 49,
    },
    "Health": {
        "Medicaid": 51,
        "Medicare": 52,
        "Govt. Child Health Ins. Programs": 53,
        "Home and Community Based Care": 54,
        "Private Health Insurance": 55,
        "Long Term Health Care Facilities": 56,
        "State and Local Health": 57,
        "Other Health": 59,
    },
    "Housing": {
        "Federally Subsidized Housing": 61,
        "Real Property/Home Ownership": 62,
        "Private Landlord/Tenant": 63,
        "Public Housing": 64,
        "Mobile Homes": 65,
        "Housing Discrimination": 66,
        "Mortgage Foreclosures": 67,
        "Mortgage Predatory Lending": 68,
        "Other Housing": 69,
    },
    "Income Maintenance": {
        "TANF/Welfare": 71,
        "Social Security": 72,
        "Food Stamps": 73,
        "SSID": 74,
        "SSI": 75,
        "Unemployment Compensation": 76,
        "Veterans Benefits": 77,
        "State and Local Income Maintenance": 78,
        "Other Income Maintenance": 79,
    },
    "Individual Rights": {
        "Immigration/Naturalization": 81,
        "Mental Health": 82,
        "Prisoners' Rights": 83,
        "Disability Rights": 84,
        "Civil Rights": 85,
        "Human Trafficking": 86,
        "Other Individual Rights": 89,
    },
    "Miscellaneous": {
        "Assistance to Non-Profit or Group": 91,
        "Indian Law": 92,
        "Traffic/Drivers License": 93,
        "Torts": 94,
        "Estate Planning/Wills/Probate": 95,
        "Advanced Directives/Power of Attorneys": 96,
        "Vacating Record": 97,
        "Criminal": 98,
        "Other": 99,
        "Municipal Legal Needs": 100,
    },
}

In [14]:
QUERY_OVERRIDES = {
    "TANF/Welfare":                      "TANF welfare cash assistance temporary family",
    "SSID":                              "SSDI social security disability insurance",
    "SSI":                               "SSI supplemental security income disability",
    "Wage Claims and Other FLSA":        "wage theft unpaid overtime FLSA labor standards",
    "Govt. Child Health Ins. Programs":  "CHIP children health insurance program",
    "Agricultural Workers Issues":       "farmworker migrant labor agricultural rights",
}

In [15]:
def build_queries(categories):
    rows = []
    for parent, subcats in categories.items():
        for subcat, code in subcats.items():
            if subcat in QUERY_OVERRIDES:
                query = QUERY_OVERRIDES[subcat]
            else:
                query = f"{parent} {subcat}".replace("/", " ").replace(".", "")
            rows.append({
                "qid":    str(code),
                "query":  query,
                "parent": parent,
                "subcat": subcat,
                "code":   code,
            })
    return pd.DataFrame(rows)

In [16]:
queries = build_queries(LEGAL_CATEGORIES)
queries.head()

,qid,query,parent,subcat,code
0,1,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1
1,2,Consumer Finance Collections Repo Garnishment,Consumer / Finance,Collections/Repo/Garnishment,2
2,3,Consumer Finance Contracts Warranties,Consumer / Finance,Contracts/Warranties,3
3,4,Consumer Finance Collection Practices Credit...,Consumer / Finance,Collection Practices/Creditor Harass.,4
4,5,Consumer Finance Predatory Lending Practices,Consumer / Finance,Predatory Lending Practices,5


In [17]:
bm25 = pt.terrier.Retriever(index, wmodel="BM25", num_results=10)
results = bm25.transform(queries[["qid", "query"]])
results = results.merge(queries[["qid", "parent", "subcat", "code"]], on="qid")
results.head()

,qid,docid,docno,rank,score,query,parent,subcat,code
0,1,38,d39,0,20.356000,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1
1,1,386,d387,1,19.965797,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1
2,1,659,d660,2,19.620707,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1
3,1,264,d265,3,19.289338,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1
4,1,74,d75,4,16.380326,Consumer Finance Bankruptcy Debtor Relief,Consumer / Finance,Bankruptcy/Debtor Relief,1


In [ ]:
def filepath_to_url(filepath):
    stem = Path(filepath).stem  # strip .html
    url  = stem.replace("https___", "https://", 1).replace("http___", "http://", 1)
    url  = re.sub(r'(https?://[^/]+)_', r'\1/', url, count=1)
    url  = url.replace("_", "/")
    return url

Rebuilt url_map with 2014 entries


{'C:\\Users\\Jia Chen\\Downloads\\Projects\\CISC-portal\\server\\ml_scripts\\ML\\Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org.html': 'https://www.washingtonlawhelp.org',
 'C:\\Users\\Jia Chen\\Downloads\\Projects\\CISC-portal\\server\\ml_scripts\\ML\\Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_.html': 'https://www.washingtonlawhelp.org/',
 'C:\\Users\\Jia Chen\\Downloads\\Projects\\CISC-portal\\server\\ml_scripts\\ML\\Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_accessibility-statement.html': 'https://www.washingtonlawhelp.org/accessibility-statement',
 'C:\\Users\\Jia Chen\\Downloads\\Projects\\CISC-portal\\server\\ml_scripts\\ML\\Crawled Contents\\washingtonlawhelp\\https___www.washingtonlawhelp.org_cdn-cgi_l_email-protection.html': 'https://www.washingtonlawhelp.org/cdn-cgi/l/email-protection',
 'C:\\Users\\Jia Chen\\Downloads\\Projects\\CISC-portal\\server\\ml_scripts\\ML\\Crawled Contents\\washingtonlaw

In [23]:
def filepath_to_url(filepath):
    stem = Path(filepath).stem  # strip .html
    url  = stem.replace("https___", "https://", 1).replace("http___", "http://", 1)
    url  = re.sub(r'(https?://[^/]+)_', r'\1/', url, count=1)
    url  = url.replace("_", "/")
    return url

In [25]:
meta = index.getMetaIndex()

output = {}
for _, row in results.iterrows():
    parent = row["parent"]
    subcat = row["subcat"]

    docid    = int(row["docno"][1:])
    filepath = meta.getItem("filename", docid)
    real_url = filepath_to_url(filepath)

    output.setdefault(parent, {}).setdefault(subcat, {"code": row["code"], "pages": []})
    output[parent][subcat]["pages"].append(real_url)

with open("resource.json", "w") as f:
    json.dump(output, f, indent=2)

output

{'Consumer / Finance': {'Bankruptcy/Debtor Relief': {'code': 1,
   'pages': ['https://www.washingtonlawhelp.org/en/billing-and-medicaid-apple-health',
    'https://www.washingtonlawhelp.org/en/what-do-i-do-if-my-loved-one-just-went-jail',
    'http://app.leg.wa.gov/RCW/default.aspx/cite/6.13',
    'https://www.washingtonlawhelp.org/en/supported-decision-making',
    'https://www.washingtonlawhelp.org/en/divorce-basics',
    'https://app.leg.wa.gov/rcw/default.aspx/Cite/35A',
    'http://app.leg.wa.gov/RCW/default.aspx/cite/6.32',
    'http://app.leg.wa.gov/RCW/default.aspx/cite/6.36',
    'http://app.leg.wa.gov/RCW/default.aspx/cite/79.24.520',
    'http://app.leg.wa.gov/RCW/default.aspx/cite/6.17.190']},
  'Collections/Repo/Garnishment': {'code': 2,
   'pages': ['https://www.washingtonlawhelp.org/en/what-do-i-do-if-my-loved-one-just-went-jail',
    'https://www.washingtonlawhelp.org/en/billing-and-medicaid-apple-health',
    'https://www.washingtonlawhelp.org/en/bankruptcy',
    'http